In [1]:
import numpy as np

from numba import njit

from scipy.integrate import solve_ivp

from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, accuracy_score

import plotly.graph_objects as go

import fastplotlib as fpl

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

In [4]:
time = 100
dt = 0.005
steps = int(time / dt)

time = 100
dt = 0.005

# Transient for washing out start of lorentz
# Transient for washing out fresh springs
steps = int(time / dt)
transient_steps_lorentz = int(steps * 0.1)
transient_steps_springs = int(steps * 0.1)
total_steps = steps + transient_steps_lorentz + transient_steps_springs

test_size = 0.2

t = np.linspace(0, total_steps * dt, total_steps)

In [5]:
sigma, rho, beta = 10, 28, 8.0 / 3.0

initial_state = [1.0, 1.0, 1.0]

def lorenz_system(t, state, sigma=sigma, rho=rho, beta=beta):
    x, y, z = state
    dx = sigma * (y - x)
    dy = x * (rho - z) - y
    dz = x * y - beta * z
    return [dx, dy, dz]

sol = solve_ivp(
    lorenz_system,
    (0, total_steps * dt),
    initial_state,
    t_eval=t,
)

lorentz_dataset = sol.y.T[transient_steps_lorentz:]

mean_l = np.mean(lorentz_dataset, axis=0)
std_l = np.std(lorentz_dataset, axis=0)
lorentz_scaled = (lorentz_dataset - mean_l) / std_l

In [6]:
def lorentz_plot(data):
    fig = go.Figure(
        data=go.Scatter3d(
            x=data[:, 0],
            y=data[:, 1],
            z=data[:, 2],
            mode="lines",
            line=dict(color="blue", width=2),
        )
    )
    return fig

In [7]:
fig = lorentz_plot(lorentz_dataset)
fig.show()

In [8]:
def create_stiffness_matrix(node_positions, connections):
    num_nodes = node_positions.shape[0]
    dims = node_positions.shape[1]
    K = np.zeros((num_nodes * dims, num_nodes * dims))

    for conn in connections:
        node_conn = conn[:2].astype(int)
        node_pos = node_positions[node_conn]
        k_val = conn[2]

        diff_vec = np.diff(node_pos, axis=0).flatten()
        unit_dir = diff_vec / np.linalg.norm(diff_vec)

        sub_block = np.outer(unit_dir, unit_dir)
        k_local = k_val * np.block([[sub_block, -sub_block], [-sub_block, sub_block]])

        global_indices = (node_conn * dims + np.arange(dims)[:, None]).flatten("F")

        for local_row, global_row in enumerate(global_indices):
            for local_col, global_col in enumerate(global_indices):
                K[global_row, global_col] += k_local[local_row, local_col]

    return K

In [9]:
@njit
def run_simulation(steps, dt, matrix_size, M_INV, C, K, U):
    x = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    for i in range(1, steps):
        acc = M_INV @ (-K @ x[i - 1] - C @ v[i - 1] + U[i - 1])

        x[i] = x[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        acc_next = M_INV @ (-K @ x[i] - C @ (v[i - 1] + acc * dt) + U[i])

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return x, v

In [10]:
def ridge_regression(X, Y, random_split=True, test_size=test_size):
    scaler = StandardScaler()

    if random_split:
        X_train, X_test, Y_train, Y_test = train_test_split(
            X, Y, test_size=test_size, random_state=42
        )
    else:
        X_train, X_test = X[:int(len(X) * (1 - test_size))], X[int(len(X) * (1 - test_size)):]
        Y_train, Y_test = Y[:int(len(Y) * (1 - test_size))], Y[int(len(Y) * (1 - test_size)):]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    model = RidgeCV()
    model.fit(X_train_scaled, Y_train)
    Y_pred = model.predict(X_test_scaled)

    return model, (Y_test, Y_pred), scaler

In [11]:
def spring_animation(
    disp, nodes_pos, connections_list, size=15, external=False, is_3d=False, frames_moved=5, max_frames=2000, animate=True
):
    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(cameras="3d", controller_types="orbit", canvas="glfw" if external else "jupyter")
    )

    coords = nodes_pos_3d + disp_3d[0]
    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors="magenta"
    )

    lines = [
        fig[0, 0].add_line(
            data=np.vstack(
                [coords[int(row[0])], coords[int(row[1])]]
            ).astype(np.float32),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    frame_tracker = 0
    test = True
    def update_springs(canvas):
        nonlocal frame_tracker, test
        # if not test:
        #     return
        # test = False
        frame_tracker = (frame_tracker + frames_moved) % steps

        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]
        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

In [12]:
def weight_plot(weights):
    labels = [
        f"{'Pos' if i % 2 == 0 else 'Vel'} Node {i//2 + 1}" for i in range(len(weights))
    ]

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=weights,
                marker_color=np.where(weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Spring/Mass Node",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

# Test Chain with U from -1 to 1

In [12]:
N = 10
nodes_pos = np.arange(0, N).reshape(-1, 1)
node_ids = np.arange(N)

In [13]:
u_val = 1
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [14]:
rng = np.random.default_rng(42)

tau_steps = int(.7 / dt)

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N-1)

transient_steps = transient_steps_lorentz + transient_steps_springs
U = np.zeros((steps + transient_steps, matrix_size))
U[:, 0] = u

In [15]:
connections_list = np.column_stack((node_ids[:-1], node_ids[1:], k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [16]:
disp, v = run_simulation(steps + transient_steps, dt, matrix_size, M_INV, DAMP, K, U)
disp = disp[transient_steps:]
v = v[transient_steps:]
Y = u[transient_steps:]

In [17]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[tau_steps:], Y[:-tau_steps])

In [18]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9124 0.2959


In [19]:
y_pred_discrete = np.where(Y_pred > 0.0, u_val, -u_val)
r_2 = r2_score(Y_test, Y_pred)
accuracy = accuracy_score(Y_test, y_pred_discrete) * 100
print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9124 99.45%


In [20]:
fig = weight_plot(model.coef_)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list)
fig.show()

RFBOutputContext()

# Chain with Lorentz

In [12]:
N = 10
nodes_pos = np.arange(0, N).reshape(-1, 1)
node_ids = np.arange(N)

In [13]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N - 1)

U = np.zeros((steps + transient_steps_springs, matrix_size))
U[:, 0] = lorentz_scaled[:, 0]
U[:, int(N/2)] = lorentz_scaled[:, 1]
U[:, -1] = lorentz_scaled[:, 2]

In [14]:
connections_list = np.column_stack((node_ids[:-1], node_ids[1:], k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [15]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
u = lorentz_scaled[transient_steps_springs:]

In [17]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], u[tau_steps:], False)

In [146]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.5058 0.6855


In [147]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list)
fig.show()

RFBOutputContext()

plot r^2 real vs predicted

## Is it better to use random or sequential test split

In [ ]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], u[tau_steps:], True)

In [ ]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.8638 0.1401


Seems to get a better score but what about against the future or a sequential split

In [ ]:
scaler = StandardScaler()

X_data = X[:-tau_steps]
u_data = u[tau_steps:]
X_train, X_test = (
    X_data[: int(len(X_data) * (1 - test_size))],
    X_data[int(len(X_data) * (1 - test_size)) :],
)
Y_train, Y_test = (
    u_data[: int(len(u_data) * (1 - test_size))],
    u_data[int(len(u_data) * (1 - test_size)) :],
)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
Y_pred = model.predict(X_test_scaled)

In [ ]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.6703 0.3256


In [ ]:
fig = lorentz_plot(Y_pred)
fig.show()

So use sequential split

# Test Ring with U from -1 to 1

In [148]:
N = 30

theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
radius = 10.0

x_pos = radius * np.cos(theta)
y_pos = radius * np.sin(theta)

nodes_pos = np.column_stack((x_pos, y_pos))

In [149]:
u_val = 2
u = (t.astype(int) % 2) * 2 * u_val - u_val

In [150]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 100, size=N)

transient_steps = transient_steps_lorentz + transient_steps_springs

U = np.zeros((steps + transient_steps, matrix_size))
target_node = 0
target_pos = np.array([x_pos[target_node], y_pos[target_node]])
target_tan_vec = np.array([-target_pos[1], target_pos[0]]) / np.linalg.norm(target_pos)
U[:, 0:2] = np.outer(u, target_tan_vec)

In [151]:
node_ids = np.arange(N)
connections_list = np.column_stack((node_ids, np.roll(node_ids, -1), k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [152]:
disp, v = run_simulation(steps + transient_steps, dt, matrix_size, M_INV, DAMP, K, U)
disp = disp[transient_steps:]
v = v[transient_steps:]
Y = u[transient_steps:]

In [153]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[tau_steps:], Y[:-tau_steps])

In [154]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9783 0.2947


In [155]:
y_pred_discrete = np.where(Y_pred > 0.0, u_val, -u_val)

r_2 = r2_score(Y_test, Y_pred)
accuracy = accuracy_score(Y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.9783 99.45%


In [156]:
fig = weight_plot(model.coef_)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=False)
fig.show()

RFBOutputContext()

# Ring with Lorentz

## Pure Ring

In [47]:
N = 30

theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
radius = 10.0
x_pos = radius * np.cos(theta)
y_pos = radius * np.sin(theta)

nodes_pos = np.column_stack((x_pos, y_pos))

In [48]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 8, size=N)

U = np.zeros((steps + transient_steps_springs, matrix_size))
target_nodes = np.array([0, int(N / 3), int(2 * N / 3)])
target_positions = nodes_pos[target_nodes]
target_tan_vecs = np.column_stack((-target_positions[:, 1], target_positions[:, 0]))
norms = np.linalg.norm(target_positions, axis=1, keepdims=True)
target_tan_vecs = target_tan_vecs / norms
for i, node_idx in enumerate(target_nodes):
    col_start = node_idx * dims
    U[:, col_start : col_start + 2] = np.outer(
        lorentz_scaled[:, i], target_tan_vecs[i]
    )

In [49]:
node_ids = np.arange(N)
connections_list = np.column_stack((node_ids, np.roll(node_ids, -1), k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [50]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
Y = lorentz_scaled[transient_steps_springs:]

In [51]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(
    X[:-tau_steps], Y[tau_steps:], False
)

In [52]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9132 0.2423


In [ ]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True)
fig.show()

: 

## Bike Ring

In [165]:
N = 100

theta = np.linspace(0, 2 * np.pi, N, endpoint=False)
radius = 10.0
x_pos = radius * np.cos(theta)
y_pos = radius * np.sin(theta)

nodes_pos = np.column_stack((x_pos, y_pos))

In [166]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

U = np.zeros((steps + transient_steps_springs, matrix_size))
target_nodes = np.array([0, int(N / 3), int(2 * N / 3)])
target_positions = nodes_pos[target_nodes]
target_tan_vecs = np.column_stack((-target_positions[:, 1], target_positions[:, 0]))
norms = np.linalg.norm(target_positions, axis=1, keepdims=True)
target_tan_vecs = target_tan_vecs / norms
for channel, node_idx in enumerate(target_nodes):
    col_start = node_idx * dims
    U[:, col_start : col_start + 2] = np.outer(
        lorentz_scaled[:, channel], target_tan_vecs[channel]
    )

In [167]:
rng = np.random.default_rng(42)

node_ids = np.arange(N)
k_vals = rng.uniform(0.5, 8, size=N)
ring_connections = np.column_stack((node_ids, np.roll(node_ids, -1), k_vals))

half_N = N // 2
src_nodes = np.arange(half_N)
dst_nodes = src_nodes + half_N
k_vals = rng.uniform(400, 800, size=half_N)
cross_connections = np.column_stack((src_nodes, dst_nodes, k_vals))

k_vals = rng.uniform(0.5, 8, size=N)
shift = int(N * .2)
diag_connections = np.column_stack((node_ids, np.roll(node_ids, shift), k_vals))

connections_list = np.vstack((ring_connections, cross_connections, diag_connections))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [168]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
Y = lorentz_scaled[transient_steps_springs:]

In [169]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], Y[tau_steps:], False)
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9880 0.0842


In [170]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True)
fig.show()

# Grid

In [ ]:
N = 10

x = np.arange(N)
y = np.arange(N)

xx, yy = np.meshgrid(x, y)

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [ ]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

U = np.zeros((steps + transient_steps_springs, matrix_size))
# target_nodes = np.array([0, int(num_nodes / 3), int(2 * num_nodes / 3)])
target_nodes = rng.integers(low=0, high=num_nodes, size=(3))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(lorentz_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [279]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_nodes = np.concatenate(
    [
        node_ids[:, :-1].flatten(),  # Right links source
        node_ids[:-1, :].flatten(),  # Down links source
        node_ids[:-1, :-1].flatten(),  # Down-Right diagonals source
        node_ids[:-1, 1:].flatten(),  # Down-Left diagonals source
    ]
)

dst_nodes = np.concatenate(
    [
        node_ids[:, 1:].flatten(),  # Right links destination
        node_ids[1:, :].flatten(),  # Down links destination
        node_ids[1:, 1:].flatten(),  # Down-Right diagonals destination
        node_ids[1:, :-1].flatten(),  # Down-Left diagonals destination
    ]
)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [280]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
Y = lorentz_scaled[transient_steps_springs:]

In [281]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], Y[tau_steps:], False)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9782 0.1206


In [282]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True)
fig.show()

## Triangle Lattice

In [13]:
N = 10

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [14]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

U = np.zeros((steps + transient_steps_springs, matrix_size))
# target_nodes = np.array([0, int(num_nodes / 3), int(2 * num_nodes / 3)])
target_nodes = rng.integers(low=0, high=num_nodes, size=(3))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(lorentz_scaled, 2, axis=1)
U[:, col_indices] = vectorized_force

In [15]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_nodes = np.concatenate(
    [
        node_ids[:, :-1].flatten(),  # Right links source
        node_ids[:-1, :].flatten(),  # Down links source
        node_ids[:-1:2, :-1].flatten(),  # Down-Right diagonals source
        node_ids[1:-1:2, 1:].flatten(),  # Down-Left diagonals source
    ]
)

dst_nodes = np.concatenate(
    [
        node_ids[:, 1:].flatten(),  # Right links destination
        node_ids[1:, :].flatten(),  # Down links destination
        node_ids[1::2, 1:].flatten(),  # Down-Right diagonals destination
        node_ids[2::2, :-1].flatten(),  # Down-Left diagonals destination
    ]
)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [17]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
Y = lorentz_scaled[transient_steps_springs:]

In [18]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], Y[tau_steps:], False)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.9719 0.1383


In [19]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True)
fig.show()

2026-07-21 15:41:24.290 python[98805:7786952] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-21 15:41:24.290 python[98805:7786952] +[IMKInputSession subclass]: chose IMKInputSession_Modern


## Triangle Lattice

In [22]:
from dysts.maps import Henon

In [31]:
steps = 19999

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [32]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

In [33]:
N = 10

x = np.arange(N, dtype=float)
y = np.arange(N, dtype=float) * np.sqrt(3) / 2

xx, yy = np.meshgrid(x, y)
xx[::2] = xx[::2] + 0.5

nodes_pos = np.column_stack((xx.flatten(), yy.flatten()))

In [34]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

U = np.zeros((steps + transient_steps_springs, matrix_size))
# target_nodes = np.array([0, int(num_nodes / 3), int(2 * num_nodes / 3)])
target_nodes = rng.integers(low=0, high=num_nodes, size=(3))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
vectorized_force = np.repeat(henon_scaled, 3, axis=1)
U[:, col_indices] = vectorized_force

In [35]:
node_ids = np.arange(x.size * y.size).reshape(y.size, x.size)

src_nodes = np.concatenate(
    [
        node_ids[:, :-1].flatten(),  # Right links source
        node_ids[:-1, :].flatten(),  # Down links source
        node_ids[:-1:2, :-1].flatten(),  # Down-Right diagonals source
        node_ids[1:-1:2, 1:].flatten(),  # Down-Left diagonals source
    ]
)

dst_nodes = np.concatenate(
    [
        node_ids[:, 1:].flatten(),  # Right links destination
        node_ids[1:, :].flatten(),  # Down links destination
        node_ids[1::2, 1:].flatten(),  # Down-Right diagonals destination
        node_ids[2::2, :-1].flatten(),  # Down-Left diagonals destination
    ]
)

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [40]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U*5
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
Y = henon_scaled[transient_steps_springs:]

In [41]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], Y[tau_steps:], False)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.4039 0.7580


In [44]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
# fig = lorentz_plot(Y_pred)
# fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True)
fig.show()

# Cube

In [34]:
N = 3

x = np.arange(N)
y = np.arange(N)
z = np.arange(N)

xx, yy, zz = np.meshgrid(x, y, z)

nodes_pos = np.column_stack((xx.flatten(), yy.flatten(), zz.flatten()))

In [35]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

U = np.zeros((steps + transient_steps_springs, matrix_size))
# Used to do 3 equal nodes apart but grid is kinda wierd so we doing random np.array([0, int(num_nodes / 3), int(2 * num_nodes / 3)])
target_nodes = rng.integers(low=0, high=num_nodes, size=(3))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)
# np.repeat(lorentz_scaled, dims, axis=1) pushes a single node in xyz with force of x, y, and z of lorentz
# np.tile(lorentz_scaled, (1, dims)) pushes all nodes in direction of the xyz of lorentz
vectorized_force = np.repeat(lorentz_scaled, dims, axis=1)
U[:, col_indices] = vectorized_force

In [36]:
node_ids = np.arange(z.size * y.size * x.size).reshape(z.size, y.size, x.size)

# 13 direction 😭
src_slices = [
    # 1D Straight Axes
    np.s_[:, :, :-1],  # Right (+x)
    np.s_[:, :-1, :],  # Down (+y)
    np.s_[:-1, :, :],  # Deep (+z)
    #2D Planar Diagonals
    np.s_[:, :-1, :-1],  # Down-Right (+y, +x)
    np.s_[:, 1:, :-1],  # Up-Right (-y, +x)
    np.s_[:-1, :, :-1],  # Deep-Right (+z, +x)
    np.s_[1:, :, :-1],  # Shallow-Right (-z, +x)
    np.s_[:-1, :-1, :],  # Deep-Down (+z, +y)
    np.s_[:-1, 1:, :],  # Deep-Up (+z, -y)
    #3D Spatial Corner Diagonals
    np.s_[:-1, :-1, :-1],  # Deep-Down-Right (+z, +y, +x)
    np.s_[:-1, 1:, :-1],  # Deep-Up-Right (+z, -y, +x)
    np.s_[1:, :-1, :-1],  # Shallow-Down-Right (-z, +y, +x)
    np.s_[1:, 1:, :-1],  # Shallow-Up-Right (-z, -y, +x)
]

dst_slices = [
    #1D Straight Axes
    np.s_[:, :, 1:],  # Right
    np.s_[:, 1:, :],  # Down
    np.s_[1:, :, :],  # Deep
    #2D Planar Diagonals
    np.s_[:, 1:, 1:],  # Down-Right
    np.s_[:, :-1, 1:],  # Up-Right
    np.s_[1:, :, 1:],  # Deep-Right
    np.s_[:-1, :, 1:],  # Shallow-Right
    np.s_[1:, 1:, :],  # Deep-Down
    np.s_[1:, :-1, :],  # Deep-Up
    #3D Spatial Corner Diagonals
    np.s_[1:, 1:, 1:],  # Deep-Down-Right
    np.s_[1:, :-1, 1:],  # Deep-Up-Right
    np.s_[:-1, 1:, 1:],  # Shallow-Down-Right
    np.s_[:-1, :-1, 1:],  # Shallow-Up-Right
]

src_nodes = np.concatenate([node_ids[sl].flatten() for sl in src_slices])
dst_nodes = np.concatenate([node_ids[sl].flatten() for sl in dst_slices])

rng = np.random.default_rng(42)
k_vals = rng.uniform(0.5, 8, size=src_nodes.shape[0])
connections_list = np.column_stack((src_nodes, dst_nodes, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

In [37]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
disp = disp[transient_steps_springs:]
v = v[transient_steps_springs:]
Y = lorentz_scaled[transient_steps_springs:]

In [38]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), scaler = ridge_regression(
    X[:-tau_steps], Y[tau_steps:], False
)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.8576 0.3431


In [41]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True, is_3d=True, max_frames=20000)
fig.show()

/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/pygfx/objects/_ruler.py:400: RuntimeWarning: divide by zero encountered in divide
  screen_full = (ndc_full[:, :2] / ndc_full[:, 3:4]) * half_canvas_size
/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/pygfx/objects/_ruler.py:400: RuntimeWarning: invalid value encountered in divide
  screen_full = (ndc_full[:, :2] / ndc_full[:, 3:4]) * half_canvas_size
/Users/churroc/Personal/code/reservoir_research/.venv/lib/python3.13/site-packages/pygfx/objects/_ruler.py:412: RuntimeWarning: invalid value encountered in divide
  screen_sel = (ndc_sel[:, :2] / ndc_sel[:, 3:4]) * half_canvas_size


## Open Loop

Basically instead of mapping out the lorentz values and telling it to predict the last 20 percent. From last test point I'll integrate from my training then use my linear regression to predict it. Then I'll plug in the new point and keep going. This will accumulate errors way more but don't need the ode anymore since this predicts it.

In [ ]:
steps_to_predict = int(steps * test_size)
current_step = steps - steps_to_predict - 1

# Get current state of springs kinda like the truth
current_x = disp[current_step]
current_v = v[current_step]
current_state = np.hstack([current_x, current_v]).reshape(1, -1)
current_state_scaled = scaler.transform(current_state)

# Then predict the lorentz step based of it
Y_pred_open_loop = np.zeros_like(Y_pred)
Y_pred_open_loop[0] = model.predict(current_state_scaled)

# Then app
U_current = np.zeros((matrix_size))
col_indices = (target_nodes[:, None] * dims + np.arange(dims)).reshape(-1)

for i in range(1, steps_to_predict):
    vectorized_force = np.repeat(Y_pred_open_loop[i-1], dims)
    U_current[col_indices] = vectorized_force

    acc = M_INV @ (-K @ current_x - DAMP @ current_v + U_current)
    x_next = current_x + current_v * dt + acc * 0.5 * dt**2
    acc_next = M_INV @ (-K @ x_next - DAMP @ (current_v + acc * dt) + U_current)
    v_next = current_v + 0.5 * (acc + acc_next) * dt

    current_x, current_v = x_next, v_next

    current_step += 1
    current_state = np.hstack([current_x, current_v]).reshape(1, -1)
    current_state_scaled = scaler.transform(current_state)
    Y_pred_open_loop[i] = model.predict(current_state_scaled)

2026-07-02 14:01:08.105 python[8327:284137] +[IMKClient subclass]: chose IMKClient_Modern
2026-07-02 14:01:08.105 python[8327:284137] +[IMKInputSession subclass]: chose IMKInputSession_Modern


In [18]:
r_2 = r2_score(Y_pred_open_loop, Y_pred)
mse = root_mean_squared_error(Y_pred_open_loop, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

-0.0004 6280512295842.2764


In [19]:
fig = lorentz_plot(Y_pred_open_loop)
fig.show()

# Fractal Serpeieneine Triangle Tree

In [42]:
corners = np.array([(0.0, 0.0), (1.0, 0.0), (0.5, np.sqrt(3) / 2)])


def get_triangles(p1, p2, p3, depth):
    if depth == 0:
        return [(p1, p2), (p2, p3), (p3, p1)]

    # Midpoints
    m1 = (p1 + p2) / 2
    m2 = (p2 + p3) / 2
    m3 = (p3 + p1) / 2

    # Recurse into the 3 outer sub-triangles
    return (
        get_triangles(p1, m1, m3, depth - 1)
        + get_triangles(m1, p2, m2, depth - 1)
        + get_triangles(m3, m2, p3, depth - 1)
    )


edges = get_triangles(corners[0], corners[1], corners[2], 4)

raw_nodes = []
for p1, p2 in edges:
    raw_nodes.extend([tuple(p1), tuple(p2)])

nodes_pos = np.unique(np.array(raw_nodes), axis=0)
node_to_idx = {tuple(node): i for i, node in enumerate(nodes_pos)}

connections = []
for p1, p2 in edges:
    connections.append([node_to_idx[tuple(p1)], node_to_idx[tuple(p2)]])

In [43]:
rng = np.random.default_rng(42)

tau_steps = 1

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_diag = rng.uniform(0.1, 0.5, size=matrix_size)
m_inv_diag = 1.0 / m_diag
M_INV = np.diag(m_inv_diag)

DAMP = np.diag(rng.uniform(0.1, 0.3, size=matrix_size))

k_vals = rng.uniform(0.5, 100, size=len(connections))
connections_list = np.column_stack((connections, k_vals))
K = create_stiffness_matrix(nodes_pos, connections_list)

U = np.zeros((steps + transient_steps_springs, matrix_size))
# Since last half of nodes should be closer to ends
target_nodes = rng.integers(low=int(num_nodes / 2), high=num_nodes, size=3)
for i, node_idx in enumerate(target_nodes):
    col_start = node_idx * dims
    U[:, col_start : col_start + dims] = lorentz_scaled[:, :dims] * .3

In [44]:
disp, v = run_simulation(
    steps + transient_steps_springs, dt, matrix_size, M_INV, DAMP, K, U
)
# disp = disp[transient_steps_springs:]
# v = v[transient_steps_springs:]
# Y = lorentz_scaled[transient_steps_springs:]
Y = lorentz_scaled

In [45]:
X = np.column_stack((disp, v))
model, (Y_test, Y_pred), _ = ridge_regression(X[:-tau_steps], Y[tau_steps:], False)

r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)
print(f"{r_2:.4f}", f"{mse:.4f}")

0.7071 0.3421


In [46]:
fig = weight_plot(np.linalg.norm(model.coef_, axis=0))
fig.show()
fig = lorentz_plot(Y_pred)
fig.show()
fig = spring_animation(disp, nodes_pos, connections_list, 10, external=True, frames_moved=1)
fig.show()

# Ideas

I could do a honeycomb hexagonal lattice instead of the triangle lattice

I could also a pyramid where propgation can go through layers. Maybe even a double pyramid that has like 3 top point and form a physical neural network.

Maybe use features like the kinetic energy or like a matrix of the energy of the system kinda??

Move from 3D to 4D cubes

Make a torroid on that though for the ring could we use mod maybe to try to get a ring in 1D since that would simplify code for a lot of things including torroid where math and code would only need a 2D.

Try to use bayesian optimzation or other methods to try to get spectral radius figure out or in general try to get down to just zeta and k_conn with non dimensionalization.
Saying this bad but non dimensionalize
Then once I have a random connection we modify the zeta value to see how much it modifies memory and output
Then try to use bayesian optimzation or anything to find best zeta value
But spectral radius is just me saying maybe a connection between the K stiff matrix and the K matrix we modify in RC.

Maybe instead of feature change the force from -1 to 1 with something like fourier series that are not linear but at least continuous.

mobius strip too

plot r^2 so then we could see what actual vs predicted value is and we get a better sense of what is happening